# 06 — Load Final Bridge Dataset into PostgreSQL

Controlled workflow:

```text
Notebook 04
   ↓
bridge_ml_dataset_neu.csv
   ↓
Notebook 05
   ↓
bridge_ml_dataset_imputed.csv
   ↓
Notebook 06
   ↓
PostgreSQL final.bridge_ml_dataset_final
```

This notebook does **not** perform imputation. It loads the canonical imputed dataset from Notebook 05 into PostgreSQL and validates the database table after loading.

The original NEU dataset and the local imputed CSV are preserved.


In [1]:
import os
import pandas as pd
from pathlib import Path
from getpass import getpass
from sqlalchemy import create_engine, URL, text


In [2]:
# ============================================================
# PORTABLE PROJECT CONFIGURATION
# ============================================================

def find_project_root():
    env_root = os.getenv("BRIDGE_PROJECT_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / "Dataset_PlanA-B").exists():
            return root
        raise FileNotFoundError(
            f"BRIDGE_PROJECT_ROOT does not contain Dataset_PlanA-B: {root}"
        )

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Dataset_PlanA-B").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Set BRIDGE_PROJECT_ROOT to the project folder."
    )

PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"

# Canonical input produced by Notebook 05
INPUT_DIR = OUTPUT_ROOT / "05_Dataset_Imputation"
IMPUTED_CSV = INPUT_DIR / "bridge_ml_dataset_imputed.csv"

# Notebook 06 audit/provenance outputs
OUTPUT_DIR = OUTPUT_ROOT / "06_Load_Final_Bridge_Dataset_to_PostgreSQL"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_INVENTORY_FILE = OUTPUT_DIR / "bridge_ml_dataset_final_data_inventory.csv"
DATA_MANIFEST_FILE = OUTPUT_DIR / "06_data_manifest.txt"

# PostgreSQL destination
SCHEMA = "final"
FINAL_TABLE = "bridge_ml_dataset_final"

print("Project root:", PROJECT_ROOT)
print("Input:", IMPUTED_CSV)
print("PostgreSQL destination:", f"{SCHEMA}.{FINAL_TABLE}")
print("Audit output:", OUTPUT_DIR)


Project root: C:\Datenanalyse\final Project
Input: C:\Datenanalyse\final Project\Output_PlanA-B\05_Dataset_Imputation\bridge_ml_dataset_imputed.csv
PostgreSQL destination: final.bridge_ml_dataset_final
Audit output: C:\Datenanalyse\final Project\Output_PlanA-B\06_Load_Final_Bridge_Dataset_to_PostgreSQL


## 1A. DATA SOURCE / INPUT–OUTPUT MANIFEST

| Layer | Source / origin | Transfer method | Role in Notebook 06 | Destination |
|---|---|---|---|---|
| Imputed dataset | Notebook 05 `bridge_ml_dataset_imputed.csv` | Local CSV read | Canonical input | pandas `df` |
| Structural QC | pandas `df` | In-memory validation | Validate bridge-level structure before load | — |
| PostgreSQL | Database `Final_Project` | SQLAlchemy / psycopg2 | Load final dataset | `final.bridge_ml_dataset_final` |
| Database QC | PostgreSQL table | SQL queries | Validate rows, IDs and NULLs after load | — |
| Data inventory | Loaded dataframe | Generated from actual dataframe | Column-level audit | `bridge_ml_dataset_final_data_inventory.csv` |
| Manifest | Notebook 06 configuration | Generated text file | Source/transfer documentation | `06_data_manifest.txt` |

### Transfer chain

```text
Notebook 04
    ↓
Output_PlanA-B/04_Integrated_Dataset/
bridge_ml_dataset_neu.csv
    ↓
Notebook 05 — Imputation
    ↓
Output_PlanA-B/05_Dataset_Imputation/
bridge_ml_dataset_imputed.csv
    ↓
Notebook 06
    ↓
PostgreSQL: final.bridge_ml_dataset_final
```

**Notebook 06 does not download BASt, DWD or Traffic data. It does not perform imputation. It receives the already integrated and imputed dataset from Notebook 05.**

**Database loading policy:** the target table `final.bridge_ml_dataset_final` is replaced by the current canonical Notebook 05 dataset (`if_exists="replace"`). No upstream CSV is deleted or overwritten.


In [3]:
# Load the canonical imputed dataset from Notebook 05

if not IMPUTED_CSV.exists():
    raise FileNotFoundError(
        "Input dataset not found. Run Notebook 05 first: "
        f"{IMPUTED_CSV}"
    )

df = pd.read_csv(IMPUTED_CSV, low_memory=False)

print("Imputed CSV found — loading output from Notebook 05.")
print("Input:", IMPUTED_CSV)
print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("Missing cells:", int(df.isna().sum().sum()))


Imputed CSV found — loading output from Notebook 05.
Input: C:\Datenanalyse\final Project\Output_PlanA-B\05_Dataset_Imputation\bridge_ml_dataset_imputed.csv
Rows: 52,214
Columns: 97
Missing cells: 0


In [4]:
# Pre-load structural checks

assert len(df) > 0
assert "bridge_id" in df.columns
assert df["bridge_id"].notna().all()
assert df["bridge_id"].duplicated().sum() == 0
assert df.columns.is_unique

print("Duplicate bridge_id:", int(df["bridge_id"].duplicated().sum()))
print("NULL bridge_id:", int(df["bridge_id"].isna().sum()))
print("Missing cells:", int(df.isna().sum().sum()))
print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("PRE-LOAD STRUCTURAL CHECKS: PASS")


Duplicate bridge_id: 0
NULL bridge_id: 0
Missing cells: 0
Rows: 52,214
Columns: 97
PRE-LOAD STRUCTURAL CHECKS: PASS


In [5]:
# ============================================================
# POSTGRESQL CONNECTION
# ============================================================

DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "Final_Project"
DB_USER = "postgres"
DB_PASSWORD = getpass("PostgreSQL password: ")

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = create_engine(
    connection_url,
    connect_args={"connect_timeout": 10}
)

with engine.connect() as conn:
    current_db = conn.execute(text("SELECT current_database()")).scalar()

print("PostgreSQL connection: PASS")
print("Current database:", current_db)


PostgreSQL connection: PASS
Current database: Final_Project


In [6]:
# ============================================================
# LOAD INTO CANONICAL FINAL TABLE
# ============================================================

with engine.begin() as conn:
    conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{SCHEMA}"'))

df.to_sql(
    FINAL_TABLE,
    engine,
    schema=SCHEMA,
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=2000
)

print(f"Loaded {len(df):,} rows into {SCHEMA}.{FINAL_TABLE}")
print("Target table policy: REPLACE current contents with Notebook 05 canonical dataset.")


Loaded 52,214 rows into final.bridge_ml_dataset_final
Target table policy: REPLACE current contents with Notebook 05 canonical dataset.


In [7]:
# ============================================================
# POST-LOAD DATABASE VALIDATION
# ============================================================

with engine.connect() as conn:
    row_count = conn.execute(
        text(f'SELECT COUNT(*) FROM "{SCHEMA}"."{FINAL_TABLE}"')
    ).scalar()

    duplicate_ids = conn.execute(
        text(
            f'SELECT COUNT(*) FROM ('
            f'SELECT bridge_id FROM "{SCHEMA}"."{FINAL_TABLE}" '
            f'GROUP BY bridge_id HAVING COUNT(*) > 1'
            f') x'
        )
    ).scalar()

    null_ids = conn.execute(
        text(
            f'SELECT COUNT(*) FROM "{SCHEMA}"."{FINAL_TABLE}" '
            f'WHERE bridge_id IS NULL'
        )
    ).scalar()

    db_columns = conn.execute(
        text(
            f'SELECT COUNT(*) '
            f'FROM information_schema.columns '
            f'WHERE table_schema = :schema AND table_name = :table'
        ),
        {"schema": SCHEMA, "table": FINAL_TABLE}
    ).scalar()

print("Database rows:", row_count)
print("Duplicate bridge_id:", duplicate_ids)
print("NULL bridge_id:", null_ids)
print("Database columns:", db_columns)

assert row_count == len(df)
assert duplicate_ids == 0
assert null_ids == 0
assert db_columns == len(df.columns)

print("DATABASE VALIDATION: PASS")


Database rows: 52214
Duplicate bridge_id: 0
NULL bridge_id: 0
Database columns: 97
DATABASE VALIDATION: PASS


In [8]:
# ============================================================
# DATA INVENTORY + HUMAN-READABLE MANIFEST
# ============================================================

data_inventory = pd.DataFrame({
    "column_order": range(1, len(df.columns) + 1),
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_in_loaded_dataframe": [
        int(df[c].isna().sum()) for c in df.columns
    ],
})

data_inventory.to_csv(DATA_INVENTORY_FILE, index=False)

manifest_text = f"""Notebook 06 — Final PostgreSQL Load Data Manifest
====================================================

PROJECT_ROOT: {PROJECT_ROOT}
DATASET_ROOT: {DATASET_ROOT}
INPUT_DIR:    {INPUT_DIR}
OUTPUT_DIR:   {OUTPUT_DIR}

INPUT
-----
Notebook 05 canonical imputed dataset:
{IMPUTED_CSV}

PROCESSING
----------
CSV is loaded into pandas.
Pre-load structural checks are performed.
No imputation is performed.
No source data is downloaded.
The dataframe is loaded into PostgreSQL.

POSTGRESQL
----------
Database: {DB_NAME}
Schema:   {SCHEMA}
Table:    {FINAL_TABLE}

LOAD POLICY
-----------
The target table is replaced with the current canonical Notebook 05 dataset.
Upstream CSV files are preserved.

VALIDATION
----------
Rows in source dataframe: {len(df):,}
Rows in database table:   {row_count:,}
Columns in source dataframe: {len(df.columns):,}
Columns in database table:   {db_columns:,}
Duplicate bridge_id: {duplicate_ids}
NULL bridge_id: {null_ids}

OUTPUTS
-------
Data inventory: {DATA_INVENTORY_FILE}
Manifest:       {DATA_MANIFEST_FILE}

DOWNSTREAM
----------
PostgreSQL {SCHEMA}.{FINAL_TABLE} is the canonical database dataset
for downstream notebooks that consume the final imputed bridge dataset.
"""

DATA_MANIFEST_FILE.write_text(manifest_text, encoding="utf-8")

print("========================================")
print("FINAL DATABASE DATASET: READY")
print("========================================")
print(f"Table: {SCHEMA}.{FINAL_TABLE}")
print(f"Rows : {row_count:,}")
print(f"Cols : {len(df.columns):,}")
print("Input imputed CSV: PRESERVED")
print("Data inventory:", DATA_INVENTORY_FILE)
print("Data manifest:", DATA_MANIFEST_FILE)


FINAL DATABASE DATASET: READY
Table: final.bridge_ml_dataset_final
Rows : 52,214
Cols : 97
Input imputed CSV: PRESERVED
Data inventory: C:\Datenanalyse\final Project\Output_PlanA-B\06_Load_Final_Bridge_Dataset_to_PostgreSQL\bridge_ml_dataset_final_data_inventory.csv
Data manifest: C:\Datenanalyse\final Project\Output_PlanA-B\06_Load_Final_Bridge_Dataset_to_PostgreSQL\06_data_manifest.txt


In [9]:
print("========================================")
print("FINAL DATABASE DATASET: READY")
print("========================================")
print(f"Table: {SCHEMA}.{FINAL_TABLE}")
print(f"Rows : {row_count:,}")
print(f"Cols : {len(df.columns):,}")
print("Previous bridge_ml_dataset table: not modified by this load")
print("Original NEU CSV: PRESERVED")
print("Imputed CSV: PRESERVED")


FINAL DATABASE DATASET: READY
Table: final.bridge_ml_dataset_final
Rows : 52,214
Cols : 97
Previous bridge_ml_dataset table: not modified by this load
Original NEU CSV: PRESERVED
Imputed CSV: PRESERVED
